# 01 — Exploratory Data Analysis: Diabetes Progression

**Goal**: Explore the 10 baseline clinical measurements (`age`, `sex`, `bmi`, `bp`, `s1`–`s6`), investigate correlations with disease progression after 1 year, and establish hypotheses for model comparison.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


# Resolve project root
def get_project_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "configs" / "config.yaml").exists():
            return p
    return Path.cwd().resolve()


ROOT = get_project_root()
sys.path.insert(0, str(ROOT))
from src.paths import resolve

df = pd.read_csv(resolve("data/raw/data.csv"))
print("Dataset shape:", df.shape)
df.head()

## 1. Data Integrity & Scaling
- Notice that features (`age`, `sex`, `bmi`, `bp`, `s1`-`s6`) have already been mean-centered and scaled by standard deviation times the square root of n_samples in scikit-learn.
- Let's verify null values and data types.

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print("\nSummary statistics:")
df.describe().round(4)

## 2. Target Variable Analysis (`target`)
The target is a quantitative measure of diabetes progression one year after baseline.

In [ ]:
print("Target stats:")
print(df["target"].describe())

plt.figure(figsize=(7, 4))
plt.hist(df["target"], bins=25, edgecolor="black", alpha=0.7, color="steelblue")
plt.axvline(
    df["target"].mean(),
    color="red",
    linestyle="--",
    label=f"Mean ({df['target'].mean():.1f})",
)
plt.axvline(
    df["target"].median(),
    color="orange",
    linestyle="-",
    label=f"Median ({df['target'].median():.1f})",
)
plt.title("Distribution of Disease Progression (Target)")
plt.xlabel("Disease Progression Score (25 - 346)")
plt.ylabel("Patient Count")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Clinical Feature Correlations with Disease Progression
Which physiological markers drive disease progression most strongly?

In [ ]:
corr = df.corr()["target"].sort_values(ascending=False)
print("Correlations with Disease Progression:")
print(corr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# BMI vs Target
axes[0].scatter(df["bmi"], df["target"], alpha=0.6, color="teal")
axes[0].set_title(f"BMI vs Target (r = {corr['bmi']:.2f})")
axes[0].set_xlabel("Body Mass Index (scaled)")
axes[0].set_ylabel("Target Progression")
axes[0].grid(True, alpha=0.3)

# s5 (Serum triglycerides) vs Target
axes[1].scatter(df["s5"], df["target"], alpha=0.6, color="crimson")
axes[1].set_title(f"s5 (Serum Triglycerides) vs Target (r = {corr['s5']:.2f})")
axes[1].set_xlabel("s5 (log of serum triglycerides)")
axes[1].set_ylabel("Target Progression")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Key Findings & Modeling Strategy
1. **Dominant Signals**: `bmi` (+0.59) and `s5` (+0.57) are the strongest linear indicators of disease progression.
2. **Protective Factor**: `s3` (high-density lipoprotein / HDL) is negatively correlated (-0.39), which aligns with clinical medicine (HDL is protective).
3. **Small Sample Size (N = 442)**:
   - Deep decision trees risk overfitting on 442 rows.
   - Regularized Linear/Ridge regression provides a strong, resilient baseline.
   - We will pit Ridge Regression against Random Forest in a Champion/Challenger contest!